# Training YOLOv8: Koin / Tutup Botol

Jalankan sel dari atas ke bawah. **Sebelum mulai:** `Runtime` > `Change runtime type` > **T4 GPU** > Save.

Siapkan di Google Drive (`My Drive`):
- `sortir-yolo.zip` (zip folder proyek ini)
- *(sangat disarankan)* folder `foto_sendiri_mentah` dari `python sortir.py foto`, berisi `koin/`, `tutup_botol/`, dan `latar/`

## 1. Cek GPU

In [ ]:
!nvidia-smi

## 2. Ambil proyek dari Drive + install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/sortir-yolo
!unzip -q -o /content/drive/MyDrive/sortir-yolo.zip -d /content
%cd /content/sortir-yolo
!pip install -q -r requirements.txt
!ls

## 3. API key Roboflow

Colab: ikon kunci (Secrets) > tambah `ROBOFLOW_API_KEY` > aktifkan *Notebook access*.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    print("API key terbaca dari Secrets.")
except Exception:
    import getpass
    os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Tempel API key Roboflow: ")

## 4. Unduh dataset

In [ ]:
!python sortir.py unduh

## 5. Beri kotak otomatis pada dataset tutup botol

Memakai YOLO-World (prompt "bottle cap"). **Cek gambar preview**: kotak merah harus pas di tutup botol. Kotak yang tidak wajar sudah ditolak otomatis.

In [ ]:
!python sortir.py label tutup-botol --metode world
from IPython.display import Image, display
display(Image('outputs/preview_autolabel_tutup_botol.jpg'))

## 6. Foto sendiri (sangat disarankan)

Salin `foto_sendiri_mentah` dari Drive lalu beri kotak otomatis. Latar ramai (tangan + dinding) aman dengan `--metode world`. Folder `latar/` dipakai otomatis sebagai latar & contoh negatif. Lewati sel ini kalau belum punya fotonya.

In [ ]:
src = '/content/drive/MyDrive/foto_sendiri_mentah'
if os.path.isdir(src):
    !mkdir -p datasets_raw
    !cp -r "{src}" datasets_raw/foto_sendiri_mentah
    !python sortir.py label sendiri --metode world
    display(Image('outputs/preview_autolabel_foto_sendiri.jpg'))
else:
    print('Folder foto_sendiri_mentah tidak ada di Drive -> dilewati.')

## 7. Gabung + data sintetis

Baca bagian **CATATAN**. Cek `preview_sintetis.jpg`: kotak harus pas, dan koin & tutup botol muncul di berbagai latar.

In [ ]:
!python sortir.py gabung --sintetis 800
display(Image('outputs/preview_sintetis.jpg'))

## 8. Training

Hasil disimpan ke Drive (`MyDrive/yolo_runs/sortir`). Kalau sesi terputus: jalankan ulang langkah 2 lalu sel *Lanjutkan*.

In [ ]:
!python sortir.py latih --epochs 100 --batch 16 --model yolov8s.pt \
    --project /content/drive/MyDrive/yolo_runs --name sortir

In [ ]:
# Lanjutkan training yang terputus (HANYA kalau perlu)
# !python sortir.py latih --resume --project /content/drive/MyDrive/yolo_runs --name sortir

## 9. Lihat hasil (perhatikan confusion matrix: koin vs tutup botol)

In [ ]:
base = '/content/drive/MyDrive/yolo_runs/sortir'
for f in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    p = f'{base}/{f}'
    if os.path.exists(p):
        display(Image(p))

## 10. Ambil model

`best.pt` ada di `MyDrive/yolo_runs/sortir/weights/best.pt`. Taruh di folder `models/` proyek, lalu `python sortir.py deteksi`.

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yolo_runs/sortir/weights/best.pt')